<h3 style="color:#6FA8DC; font-weight:bold">12. Pandas: Working with JSON and SQL</h3>

In this notebook, we will learn how to load data from:

1. JSON files
2. JSON URLs / APIs
3. Nested JSON responses
4. SQL databases using MySQL
5. SQL queries with `pd.read_sql()`

⭐ **VVVV Important for real-world Data Science:** Data often comes from APIs and databases, not only CSV files.

<h3 style="color:#6FA8DC; font-weight:bold">PART A — WORKING WITH JSON</h3>

<h5 style="color:#78B89A; font-weight:bold;">1. What is JSON? → JavaScript Object Notation</h5>

JSON is a lightweight format used to store and exchange data.

It is commonly used by:

- APIs
- websites
- backend servers
- databases
- web applications

JSON supports:

- objects → key-value pairs
- arrays → lists
- strings
- numbers
- booleans: `true` / `false`
- null values

Example:

```json
{
    "name": "Dev",
    "age": 20,
    "is_student": true
}
```

<h5 style="color:#78B89A; font-weight:bold;">2. Import Pandas → starting point</h5>

```python
import pandas as pd
```

In [ ]:
import pandas as pd

<h5 style="color:#78B89A; font-weight:bold;">3. Opening a local JSON dataset → `read_json()`</h5>

Pandas provides:

```python
pd.read_json()
```

### Syntax

```python
df = pd.read_json("file.json")
```

If the JSON file is inside a datasets folder:

```python
df = pd.read_json("datasets/file.json")
```

This is similar to:

```python
pd.read_csv("file.csv")
```

In [ ]:
# Example
# df = pd.read_json("dataset.json")
# df.head()

<h5 style="color:#78B89A; font-weight:bold;">4. Opening JSON from a URL → `read_json()`</h5>

Pandas can directly read JSON data from a URL.

### Syntax

```python
df = pd.read_json("JSON_URL")
```

Example:

```python
url = "https://api.example.com/events"
df = pd.read_json(url)
```

⚠️ The URL must return valid JSON in a structure Pandas can interpret directly.

In [ ]:
# Example
# url = "https://example.com/data.json"
# df = pd.read_json(url)

<h3 style="color:#6FA8DC; font-weight:bold">WORKING WITH API-STYLE JSON</h3>

<h5 style="color:#78B89A; font-weight:bold;">5. Understanding API response structure → object + data array</h5>

A common API response looks like this:

```json
{
    "success": true,
    "data": [
        {
            "_id": "6ab22f36c72a086484fd1c4c",
            "title": "GFG AI FILMATHON 2026",
            "date": "2026-09-29T00:00:00.000Z",
            "description": "An AI filmmaking challenge",
            "location": "Auditorium, F-block BVCOE",
            "time": "09:30 AM onwards",
            "targetAudience": "All Students",
            "otherLinks": "[...]",
            "otherDocs": "",
            "faqs": [],
            "removedFromListAt": null,
            "createdAt": "2026-09-22T07:33:10.502Z",
            "updatedAt": "2026-09-22T09:34:06.872Z",
            "__v": 0
        }
    ]
}
```

Important:

- `success` is a boolean.
- `data` is a list of event objects.
- Each event object becomes one row.
- Keys become columns.
- `null` becomes a missing value (`NaN` in many Pandas operations).

Because the actual records are inside `data`, we should extract that list first.

<h5 style="color:#78B89A; font-weight:bold;">6. Opening API JSON using Python `requests` → extract the data list</h5>

For API responses, a common workflow is:

```python
import requests

response = requests.get(url)
json_data = response.json()

df = pd.DataFrame(json_data["data"])
```

Explanation:

- `requests.get(url)` → sends a GET request
- `.json()` → converts the response into Python dictionaries/lists
- `json_data["data"]` → extracts the list of records
- `pd.DataFrame(...)` → converts records into a DataFrame

⭐ **VVVV Important:** `pd.read_json(url)` is convenient for tabular JSON, but API responses often need `response.json()` and `pd.DataFrame()`.

In [ ]:
# Example API workflow

# import requests

# response = requests.get(url)
# json_data = response.json()

# df = pd.DataFrame(json_data["data"])
# df.head()

<h5 style="color:#78B89A; font-weight:bold;">7. Handling JSON datatypes → boolean, null, dates and lists</h5>

JSON supports datatypes that appear in API responses.

| JSON value | Python/Pandas meaning |
|---|---|
| `true` | `True` |
| `false` | `False` |
| `null` | `None` / missing value |
| `"text"` | String |
| `123` | Integer |
| `12.5` | Float |
| `[]` | List |
| `{}` | Dictionary/object |

Example:

```python
{
    "success": true,
    "removedFromListAt": null,
    "faqs": []
}
```

After converting to Python:

```python
True
None
[]
```

Pandas may display `None` as `NaN` depending on the column's datatype.

In [ ]:
# Example JSON converted into a DataFrame

sample_json = {
    "success": True,
    "data": [
        {
            "_id": "101",
            "title": "AI Workshop",
            "date": "2026-09-29T00:00:00.000Z",
            "faqs": [],
            "removedFromListAt": None
        }
    ]
}

events = pd.DataFrame(sample_json["data"])
events

<h5 style="color:#78B89A; font-weight:bold;">8. Converting date strings into datetime → `to_datetime()`</h5>

Dates in JSON are often strings:

```text
2026-09-29T00:00:00.000Z
```

Convert them using:

```python
df["date"] = pd.to_datetime(df["date"])
```

Now we can:

- extract year/month/day
- sort by date
- filter dates
- calculate date differences

```python
df["date"].dt.date
df["date"].dt.year
df["date"].dt.month
```

In [ ]:
events["date"] = pd.to_datetime(events["date"])

events[["title", "date"]]

<h5 style="color:#78B89A; font-weight:bold;">9. Handling nested JSON → `json_normalize()`</h5>

Sometimes JSON contains dictionaries inside dictionaries.

Example:

```json
{
    "name": "Dev",
    "address": {
        "city": "Delhi",
        "country": "India"
    }
}
```

A normal DataFrame may keep `address` as a dictionary.

Pandas provides:

```python
pd.json_normalize()
```

It flattens nested JSON into tabular form.

### Syntax

```python
df = pd.json_normalize(data)
```

In [ ]:
nested_data = [
    {
        "name": "Dev",
        "address": {
            "city": "Delhi",
            "country": "India"
        }
    },
    {
        "name": "Rahul",
        "address": {
            "city": "Mumbai",
            "country": "India"
        }
    }
]

flat_df = pd.json_normalize(nested_data)
flat_df

<h5 style="color:#78B89A; font-weight:bold;">10. Flattening API response with `record_path` → nested list handling</h5>

Suppose the response is:

```python
response = {
    "success": True,
    "data": [
        {
            "title": "Event 1",
            "faqs": [
                {"question": "Who can join?"},
                {"question": "What is the duration?"}
            ]
        }
    ]
}
```

We can flatten the nested `faqs` list:

```python
pd.json_normalize(
    response["data"],
    record_path="faqs",
    meta=["title"]
)
```

- `record_path` → nested list to expand
- `meta` → parent columns to preserve

In [ ]:
response = {
    "success": True,
    "data": [
        {
            "title": "Event 1",
            "faqs": [
                {"question": "Who can join?"},
                {"question": "What is the duration?"}
            ]
        }
    ]
}

faq_df = pd.json_normalize(
    response["data"],
    record_path="faqs",
    meta=["title"]
)

faq_df

<h5 style="color:#78B89A; font-weight:bold;">11. JSON parameter revision → `read_json()`</h5>

Important parameters of `pd.read_json()`:

| Parameter | Use |
|---|---|
| `path_or_buf` | File path or URL |
| `orient` | Structure/orientation of JSON |
| `typ` | Return Series or DataFrame |
| `dtype` | Specify datatypes |
| `convert_dates` | Convert date-like columns |
| `encoding` | Character encoding |
| `lines` | Read line-delimited JSON |
| `chunksize` | Read JSON in chunks |
| `nrows` | Read limited rows |

### `lines=True`

Useful when each line is a separate JSON object:

```python
df = pd.read_json(
    "data.json",
    lines=True
)
```

This format is called **JSON Lines / NDJSON**.

<h3 style="color:#6FA8DC; font-weight:bold">PART B — WORKING WITH SQL</h3>

<h5 style="color:#78B89A; font-weight:bold;">12. What is SQL? → Structured Query Language</h5>

SQL is used to communicate with relational databases.

Examples:

- MySQL
- PostgreSQL
- SQLite
- SQL Server
- Oracle

A relational database stores data in tables made of:

- rows
- columns

Instead of loading a complete CSV file, we can request only the data we need using SQL queries.

<h5 style="color:#78B89A; font-weight:bold;">13. Why connect Pandas with SQL? → database + DataFrame</h5>

Pandas is excellent for data analysis.

SQL databases are excellent for:

- storing large amounts of data
- searching/filtering data
- managing structured records
- handling multiple users and applications

Together:

```text
SQL database
      ↓
SQL query
      ↓
Pandas DataFrame
      ↓
Data analysis / ML
```

<h3 style="color:#6FA8DC; font-weight:bold">MYSQL CONNECTOR</h3>

<h5 style="color:#78B89A; font-weight:bold;">14. Install and import `mysql.connector`</h5>

Install the connector in your terminal:

```bash
pip install mysql-connector-python
```

Import it:

```python
import mysql.connector
```

This library allows Python to connect to a MySQL database.

In [ ]:
# import mysql.connector

<h5 style="color:#78B89A; font-weight:bold;">15. Connect Python to MySQL → `mysql.connector.connect()`</h5>

### Syntax

```python
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="your_database"
)
```

Important arguments:

| Argument | Meaning |
|---|---|
| `host` | Database server address |
| `user` | MySQL username |
| `password` | MySQL password |
| `database` | Database name |
| `port` | MySQL port, usually `3306` |

⚠️ Replace the example values with your own credentials. Never share database passwords publicly.

In [ ]:
# Example connection

# import mysql.connector

# connection = mysql.connector.connect(
#     host="localhost",
#     user="root",
#     password="your_password",
#     database="your_database"
# )

# print(connection.is_connected())

<h5 style="color:#78B89A; font-weight:bold;">16. Reading SQL data using `pd.read_sql()` → main method</h5>

Pandas can execute a SQL query and directly return the result as a DataFrame.

### Syntax

```python
df = pd.read_sql(
    "SELECT * FROM table_name",
    connection
)
```

Example:

```python
df = pd.read_sql(
    "SELECT * FROM students",
    connection
)
```

⭐ **VVVV Important:** `pd.read_sql()` is used to read SQL query results into a DataFrame.

In [ ]:
# Example
# query = "SELECT * FROM students"
# df = pd.read_sql(query, connection)
# df.head()

<h5 style="color:#78B89A; font-weight:bold;">17. SQL queries with filtering → retrieve only required rows</h5>

SQL supports filtering using `WHERE`.

```sql
SELECT *
FROM students
WHERE age > 18;
```

In Pandas:

```python
query = """
SELECT *
FROM students
WHERE age > 18
"""

df = pd.read_sql(query, connection)
```

This is better than loading every row and filtering later when the database is large.

In [ ]:
# query = """
# SELECT *
# FROM students
# WHERE age > 18
# """

# df = pd.read_sql(query, connection)

<h5 style="color:#78B89A; font-weight:bold;">18. Selecting specific columns from SQL → avoid unnecessary data</h5>

```sql
SELECT name, age, city
FROM students;
```

Python:

```python
query = "SELECT name, age, city FROM students"
df = pd.read_sql(query, connection)
```

Advantages:
- less data transferred
- faster query
- lower memory usage
- cleaner DataFrame

<h5 style="color:#78B89A; font-weight:bold;">19. SQL aggregation with Pandas → `GROUP BY`</h5>

SQL can perform calculations before sending data to Pandas.

Example:

```sql
SELECT city, COUNT(*) AS total_students
FROM students
GROUP BY city;
```

Python:

```python
df = pd.read_sql(query, connection)
```

Useful SQL aggregate functions:

- `COUNT()`
- `SUM()`
- `AVG()`
- `MIN()`
- `MAX()`

<h5 style="color:#78B89A; font-weight:bold;">20. Reading SQL tables using `pd.read_sql_table()`</h5>

If you are using a supported SQLAlchemy connection, Pandas also provides:

```python
pd.read_sql_table(
    "table_name",
    con
)
```

This reads a complete table into a DataFrame.

For normal SQL queries, use:

```python
pd.read_sql()
```

For reading a complete table through SQLAlchemy, use:

```python
pd.read_sql_table()
```

<h5 style="color:#78B89A; font-weight:bold;">21. Writing a DataFrame into SQL → `to_sql()`</h5>

Pandas can also send a DataFrame into a SQL database.

With a SQLAlchemy engine:

```python
df.to_sql(
    "new_table",
    con=engine,
    if_exists="replace",
    index=False
)
```

Important:

- `name` → destination table name
- `con` → database connection/engine
- `if_exists="fail"` → error if table exists
- `if_exists="replace"` → replace existing table
- `if_exists="append"` → add rows
- `index=False` → do not store DataFrame index as a column

⭐ `read_sql()` = SQL → DataFrame

⭐ `to_sql()` = DataFrame → SQL

<h3 style="color:#6FA8DC; font-weight:bold">SQL CONNECTION WORKFLOW</h3>

```text
Import mysql.connector
        ↓
Connect to MySQL
        ↓
Write SQL query
        ↓
pd.read_sql(query, connection)
        ↓
DataFrame
        ↓
Analysis / Visualization / ML
```

Example:

```python
import pandas as pd
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="college"
)

query = "SELECT * FROM students"

df = pd.read_sql(query, connection)

df.head()
```

<h3 style="color:#6FA8DC; font-weight:bold">JSON vs CSV vs SQL</h3>

| Feature | CSV | JSON | SQL |
|---|---|---|---|
| Structure | Tabular | Nested or tabular | Relational tables |
| Common source | Files | APIs/websites | Databases |
| Pandas method | `read_csv()` | `read_json()` / `json_normalize()` | `read_sql()` |
| Supports nested data | No | Yes | Through related tables |
| Best use | Simple datasets | API responses | Large structured data |
| Filtering before loading | Limited | API-dependent | Yes, using SQL |

### Final memory tricks

- `pd.read_json()` → open JSON
- `requests.get().json()` → read API response
- `pd.DataFrame(json_data["data"])` → convert API records
- `pd.json_normalize()` → flatten nested JSON
- `mysql.connector.connect()` → connect to MySQL
- `pd.read_sql()` → SQL query to DataFrame
- `pd.read_sql_table()` → SQL table to DataFrame
- `df.to_sql()` → DataFrame to SQL

<h3 style="color:#6FA8DC; font-weight:bold">FINAL TAKEAWAY</h3>

> **CSV is commonly used for files, JSON is commonly used for APIs, and SQL is commonly used for databases. Pandas provides tools to bring all three sources into a DataFrame for analysis and Machine Learning.**